In [1]:
%pip uninstall --yes 'keras' 'tensorflow'

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import warnings
import traceback
import subprocess
warnings.simplefilter('ignore')
sys.path.insert(0, '/kaggle/input/datasets/francescosabbarese97/qwen3-5-code-interpreter')

In [3]:
def set_env_tar(input_archive, temp_dir):
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'vllm', 
        'qwen-agent[code_interpreter]',
        'flashinfer-python',
    ], check=True)

In [4]:
!rm -rf ~/.cache/flashinfer

In [5]:
set_env_tar(input_archive='/kaggle/input/notebooks/francescosabbarese97/aimo3-qwen3-5-utils/wheels.tar.gz', temp_dir='/kaggle/tmp/setup')
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
#os.environ["VLLM_USE_FLASHINFER"] = "0"
#os.environ["VLLM_CACHE_ROOT"] = "/kaggle/working/vllm_cache"
#os.environ["FLASHINFER_CACHE_DIR"] = "/kaggle/working/flashinfer_cache"
#cuda_lib_path = "/usr/local/cuda/lib64"
#stubs_path = "/usr/local/cuda/lib64/stubs"
#os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib_path}:{stubs_path}:" + os.environ.get("LD_LIBRARY_PATH", "")
#os.environ["LIBRARY_PATH"] = f"{stubs_path}:" + os.environ.get("LIBRARY_PATH", "")
#os.environ["CUDA_HOME"] = "/usr/local/cuda"
#!ln -sf /usr/local/cuda/lib64/stubs/libcuda.so /usr/local/cuda/lib64/stubs/libcuda.so.1

Looking in links: /kaggle/tmp/setup/wheels


In [6]:
import gc
import re
import copy
import math
import time
import queue
import random
import requests
import threading
import contextlib
import local_code_interpreter  # noqa
import kaggle_evaluation.aimo_3_inference_server

import pandas as pd
import polars as pl

from typing import Optional
from transformers import set_seed
from qwen_agent.agents import Assistant
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
from qwen_agent.utils.output_beautify import typewriter_print

In [7]:
class CFG:
    seed = 17
    served_model_name = 'qwen-3-5'
    model_path = '/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-35b-a3b/1'
    
    dtype = 'auto'
    kv_cache_dtype = 'fp8_e4m3' #fp8_e4m3
    calculate_kv_scales = False
    
    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17450
    server_timeout = 700

    session_timeout = 960
    jupyter_timeout = 13  

    context_tokens = 32768
    buffer_tokens = 512
    search_tokens = 32
    
    gpu_memory_utilization = 0.95    
    turns = 164                        

    attempts = 6
    max_num_batched_tokens = 8192
    max_num_seqs = attempts 
    workers = 12 
    
    top_logprobs = 5
    top_p = 0.95
    top_k = 20
    presence_penalty = 1.5
    repetition_penalty = 1.0
    
    temperatures = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
    early_stop = 3

    system_prompt = """
    You are an International Mathematical Olympiad (IMO) Competitor. 

    Your primary goal is to produce a complete and rigorously justified solution. 
    Every step in your solution must be logically sound, explicitly justified, and clearly explained. 
    A correct final answer derived from flawed, incomplete, or unjustified reasoning is considered a failure.
    
    General Principles:
    - Break complex problems into smaller parts.
    - Use examples to build intuition when appropriate, but do not rely on them as proof.
    - Look for structure (symmetry, invariants, extremal arguments, ...).
    - If an approach fails, reconsider the strategy and try another path.
    - Prefer exact reasoning over guesswork. Do not guess intermediate results.
    - Do not assume unstated conditions or introduce unsupported claims.
    - If a step cannot be fully justified, explicitly acknowledge the gap instead of proceeding.
    
    {strategy}
    If the approach is not applicable, fallback to a general one.
    
    Stay focused on the given problem. Avoid unnecessary digressions or unrelated explorations.
    
    The final answer must be a non-negative integer between 0 and 99999. 
    Please reason step by step, and put your final answer within \\boxed{{}}.
    """

    tool_prompt = """
    Use this tool to execute Python code in your chain of thought. The code will not be shown to the user. This tool should be used for internal reasoning, but not for code that is intended to be visible to the user (e.g. when creating plots, tables, or files).
    When you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. You have to use print statements to access the output.
    Code should support your mathematical reasoning, not replace it.
    """
    strategies = [
    # SMALL CASES FIRST
    """
    ### Approach: Start From Small Cases ###
    Before attempting the general solution, systematically test small values.
    - Compute the answer for n=1, 2, 3, 4, 5 using Python code.
    - Look at the sequence of results. Do you see a pattern? A periodicity? A formula?
    - Only after the pattern is clear from examples, prove it holds in general.
    - If a claim appears during your proof, verify it numerically before continuing.
    Small cases are not just warm-up — they are the map to the general solution.
    """,
    
    # BRUTE FORCE THE ANSWER FIRST
    """
    ### Approach: Brute Force Before Proving ###
    Write Python code to compute the answer directly by exhaustive search or simulation.
    - Do not start with a proof. Start with code that gives the correct answer for small inputs.
    - Once you have a verified numerical answer, use it as a target to reverse-engineer the proof.
    - If the answer is a count, enumerate all valid configurations explicitly.
    - If the answer is a construction, build it step by step in code and verify all constraints.
    The brute-force answer is ground truth. Build your mathematical argument to match it.
    """,
    
    # PARITY AND MODULAR ARITHMETIC
    """
    ### Approach: Use Parity or Modular Arithmetic ###
    Check whether a modular constraint immediately restricts the problem.
    - Compute both sides of the key equation modulo 2, 3, 4, 5, 7, 9 and see if any leads to a contradiction.
    - Check the parity (odd/even) of all quantities involved.
    - If the problem involves a sequence, check if it is eventually periodic modulo some integer.
    - Use Python to compute residues and identify the period.
    A single modular contradiction can immediately prove impossibility or uniqueness.
    """,
    
    # ASSUME THE WORST: PROOF BY CONTRADICTION
    """
    ### Approach: Proof by Contradiction ###
    Assume the opposite of what you want to prove and derive a logical impossibility.
    - State the negation of the conclusion explicitly and clearly.
    - Treat this negation as a hypothesis and apply the given constraints.
    - Derive consequences step by step. Use Python to verify arithmetic at each step.
    - Identify the first point where you reach a statement that is false or self-contradicting.
    The contradiction is the proof. Make sure the contradiction is genuine, not a computation error — verify with code.
    """,
    
    # EXTREMAL PRINCIPLE
    """
    ### Approach: Consider the Extremal Element ###
    Identify the largest, smallest, maximum, or minimum object satisfying the given conditions.
    - Define precisely what you are optimizing: the largest index, the smallest sum, the element with the most connections, etc.
    - Assume this extremal element exists and derive every constraint it must satisfy from the problem conditions.
    - Use the extremality condition itself as a powerful constraint: if the maximum element had a certain property, applying an operation would produce something larger — contradiction.
    - Verify your argument on small examples with Python before writing the final proof.
    """,
    
    # MATHEMATICAL INDUCTION
    """
    ### Approach: Prove by Induction ###
    Structure your proof as a base case followed by an inductive step.
    - Identify the induction variable (n, k, the size of a set, etc.) and state it explicitly.
    - Prove the base case(s) first. Verify with Python.
    - State the inductive hypothesis precisely: "Assume the statement holds for all values up to n."
    - Prove the inductive step: show that the statement for n implies the statement for n+1.
    - Use Python to verify the inductive step for n = 1, 2, 3, 4, 5 before writing the general argument.
    Be especially careful about "strong induction" — if the step for n+1 requires all previous cases, state this explicitly.
    """,
    
    # ALGEBRAIC MANIPULATION
    """
    ### Approach: Algebraic Manipulation and Symbolic Computation ###
    Translate all conditions into explicit algebraic equations or inequalities.
    - Introduce variables for every unknown quantity and write the constraints as formulas.
    - Use algebraic identities: AM-GM, Cauchy-Schwarz, telescoping sums, factorizations, substitutions...
    - Use Python (sympy) to expand, factor, simplify, and solve symbolic expressions...
    - Verify every algebraic step: do not assume an identity holds — prove it or verify it in code.
    If you reach a polynomial equation, find its roots with sympy and verify them against all original constraints.
    """,

    # DIVIDE INTO CASES
    """
    ### Approach: Divide Into Cases ###
    Do not try to handle all situations simultaneously — split the problem into exhaustive, non-overlapping cases.
    - Identify the key variable or condition that drives the different behaviors (e.g., n even vs odd, x > 0 vs x ≤ 0).
    - List all cases explicitly and confirm they are exhaustive (they cover every possibility) and mutually exclusive.
    - Solve each case independently. Use Python to verify the solution for each case on specific examples.
    - After solving all cases, assemble the final answer.
    A clean case split often transforms an intractable general problem into several simple ones.
    """,
    
    # REDUCTION TO A KNOWN PROBLEM
    """
    ### Approach: Reduce to a Simpler or Known Problem ###
    Before solving from scratch, ask: is this problem equivalent to one I already know how to solve?
    - Strip away the specific context and identify the abstract mathematical structure (is this really a graph coloring? a fixed-point problem? a linear recurrence?).
    - Apply a transformation that maps this problem to the simpler one.
    - Solve the simpler problem, then map the solution back.
    - Verify with Python that the transformation is valid and the mapped solution satisfies all original constraints.
    Recognizing the underlying structure is often the single most powerful step in olympiad problem-solving.
    """,
    
    # DOUBLE COUNTING 
    """
    ### Approach: Double Counting ###
    Count the same quantity in two different ways and equate the results.
    - Identify a set of pairs, triples, or incidences that can be counted from two perspectives.
    - Count from perspective A (e.g., for each row, count the elements with property P).
    - Count from perspective B (e.g., for each element with property P, count the rows it appears in).
    - Equate the two counts to obtain a non-trivial identity or inequality.
    - Verify the identity with Python on small examples before using it in the proof.
    """,
    
    # NUMBER THEORY: DIVISIBILITY AND PRIME FACTORIZATION
    """
    ### Approach: Exploit Divisibility and Prime Structure ###
    If the problem involves integers, their factors, or divisibility, work with prime factorizations.
    - Factorize all key quantities. Use Python to compute factorizations for small values.
    - Write the key condition in terms of the prime factorization: what does the condition say about the exponents of 2, 3, 5, ...?
    - Use Legendre's formula (p-adic valuation) if factorials or binomial coefficients appear.
    - Look for conditions of the form "p | f(n)" and analyze them prime by prime.
    - Check divisibility conditions modulo small primes with Python to discover which primes play a role.
    """,
]

set_seed(CFG.seed)

In [8]:
class AIMO3Solver:
    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'

        self.attempts = self.cfg.attempts
        self.temperatures = self.cfg.temperatures
        self.problems_remaining = 50
        
        assert len(self.temperatures) == self.attempts, "Different len for temperatures"
        
        self._preload_model_weights()
        self.server_process = self._start_server()
        self._wait_for_server()
 
        self.boxed_pattern = re.compile(
            r'\\(?:boxed|framebox)\{(.*?)\}',
            re.DOTALL
        )

        self.final_answer_pattern = re.compile(
            r'(?:final answer|answer)\s*(?:is|:)\s*([0-9,]+)',
            re.IGNORECASE
        )
        
        self.notebook_start_time = time.time()

    
    def _preload_model_weights(self) -> None:
        print(f'### Loading model weights from {self.cfg.model_path} into OS Page Cache...', flush=True)
        start_time = time.time()
    
        files_to_load = []
        total_size = 0
        
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=24) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'### Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n', flush=True)

    
    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable,
            "-m", "vllm.entrypoints.openai.api_server",
            "--seed", str(self.cfg.seed),
            "--model", self.cfg.model_path,
            "--served-model-name", self.cfg.served_model_name,
            "--tensor-parallel-size", "1",
            "--max-num-seqs", str(self.cfg.max_num_seqs),
            "--max-num-batched-tokens", str(self.cfg.max_num_batched_tokens),
            "--gpu-memory-utilization", str(self.cfg.gpu_memory_utilization),
            "--host", "0.0.0.0",
            "--port", str(self.port),
            "--dtype", self.cfg.dtype,
            "--kv-cache-dtype", self.cfg.kv_cache_dtype,
            "--max-model-len", str(self.cfg.context_tokens),
            "--tool-call-parser", "qwen3_coder",
            "--reasoning-parser", "qwen3",
            "--enable-auto-tool-choice",
            "--disable-log-stats",
            "--enable-prefix-caching",
            "--language-model-only",
            "--trust-remote-code",
        ]

        # When chache is quantized
        if self.cfg.calculate_kv_scales:
            cmd.append('--calculate-kv-scales')
        
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

        
    def _wait_for_server(self):
        print(f"### Waiting for vLLM server...", flush=True)
    
        start_time = time.time()
        timeout = float(self.cfg.server_timeout)
    
        while time.time() - start_time < timeout:
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open("vllm_server.log", "r") as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f"Server died with code {return_code}. Full logs:\n{logs}\n")
    
            try:
                resp = requests.get(f"http://localhost:{self.port}/health", timeout=3)
                elapsed = time.time() - start_time
                print(f"### Server is ready (took {elapsed:.2f} seconds).\n", flush=True)
                return
                
            except Exception:
                time.sleep(1)
    
        raise RuntimeError(f"Server failed to start (timeout).\n")

    def _scan_for_answer(self, text: str) -> int | None:
        # Primary extraction: \boxed{} / \framebox{}
        matches = self.boxed_pattern.findall(text)

        if matches:
            raw = matches[-1].split(',')[-1].strip()
            try:
                value = int(raw)
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        # Semantic "final answer" phrase
        matches = self.final_answer_pattern.findall(text)
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass
        return None

    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        try:
            if not logprobs_buffer:
                return float('inf')
        
            total_entropy = 0.0
            token_count = 0
        
            for top_logprobs_dict in logprobs_buffer:
                if not isinstance(top_logprobs_dict, dict):
                    continue
                
                if not top_logprobs_dict:
                    continue
                
                token_entropy = 0.0
                
                for token_str, log_prob in top_logprobs_dict.items():
                    prob = math.exp(log_prob)
                    
                    if prob > 0:
                        token_entropy -= prob * math.log2(prob)
                
                total_entropy += token_entropy
                token_count += 1
        
            if token_count == 0:
                return float('inf')
        
            return total_entropy / token_count
                
        except Exception as exc:
            return float('inf')


    def _select_answer(self, detailed_results: list) -> int:
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []
        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []
        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 4})
        display(vote_dataframe)
        
        if not scored_answers:
            print(f'\nFinal Answer: 0\n', flush=True)
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n', flush=True)
        return final_answer

    
    def _init_agent_service(self, llm_cfg: dict={}, system_prompt: str=""):
        agent = Assistant(
            name=self.cfg.served_model_name,
            llm=llm_cfg,
            system_message=system_prompt,
            function_list=['local_code_interpreter'],
        )
        return agent
        

    def _run_agent(self, agent: Assistant, input_text: str, seed: int = 11):
        messages = [{'role': 'user', 'content': input_text}]
        extra_cfg = {'seed': seed}

        all_responses = list(agent.run(messages=messages, extra_generate_cfg=extra_cfg))
            
        if not all_responses:
            return '', [], 0
    
        flat_messages = []
        for chunk in all_responses:
            if isinstance(chunk, list):
                flat_messages.extend(chunk)
            elif isinstance(chunk, dict):
                flat_messages.append(chunk)
    
        logprobs = []
        total_tokens = 0
        for msg in flat_messages:            
            if msg.get('role') != 'assistant':
                continue
    
            token_ids = msg.get('token_ids', [])
            if token_ids:
                logprobs.extend(msg.get('logprobs', []))
                total_tokens += len(token_ids)
    
            content = msg.get('content', '')
            if content:
                return content, logprobs, total_tokens
    
        return '', [], total_tokens

    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        temperature: float,
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        start_time = time.time()
        
        if (stop_event is not None and stop_event.is_set()) or time.time() > deadline:
            print(f"- Attempt index {attempt_index} | Deadline reached or stop event set", flush=True)
            return {
                'Attempt': attempt_index, 
                'Answer': None, 
                'Entropy': float('inf'),
                'Response Length': 0, 
                'Time': '00:00'
            }

        answer = None
        entropy = None
        start_time = time.time()
        total_tokens = 0
        
        try:
            attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
            
            llm_cfg = {
                'model': self.cfg.served_model_name,
                'model_type': 'qwenvl_oai',
                'model_server': 'http://localhost:8000/v1',
                'api_key': 'EMPTY',
                'generate_cfg': {
                    'use_raw_api': True,
                    'temperature': temperature,
                    'seed': attempt_seed,
                    'top_p': self.cfg.top_p,
                    'top_k': self.cfg.top_k,
                    'logprobs': self.cfg.top_logprobs > 0,                        
                    'top_logprobs': self.cfg.top_logprobs,
                    'presence_penalty': self.cfg.presence_penalty,
                    'repetition_penalty': self.cfg.repetition_penalty,
                    'extra_body': {
                        'chat_template_kwargs': {
                            'enable_thinking': True
                        },
                    },
                },
            }
    
            agent = self._init_agent_service(llm_cfg, system_prompt)
            text, logprobs, total_tokens = self._run_agent(agent, problem, attempt_seed)
            answer = self._scan_for_answer(text)
            if answer is not None:
                print(f"- Attempt index {attempt_index} | Answer available {answer}", flush=True)
            else:
                print(f"- Attempt index {attempt_index} | Answer NOT available: Text {text}", flush=True)
            entropy = self._compute_mean_entropy(logprobs)
            
        except Exception as exc:
            exc_type, exc_value, exc_traceback = sys.exc_info()
            print(f"- Attempt index {attempt_index} | Python error: {exc_type} - {exc_value} - {exc_traceback}", flush=True)

        elapsed = time.time() - start_time
        time_duration = f'{int(elapsed // 60):02d}:{int(elapsed % 60):02d}'
        
        return {
            'Attempt': attempt_index, 
            'Answer': answer,
            'Entropy': entropy, 
            'Response Length': total_tokens,
            'Time': time_duration, 
        }                
            
    def process_attempt_loop(
        self,
        attempts: int,
        problem: str,
        system_prompt: str,
        temperatures: list[float],
        workers: int,
        early_stop: int,
        deadline: float
    ):

        k = min(attempts, len(self.cfg.strategies))
        strategies = random.sample(self.cfg.strategies, k)
        tasks = [(system_prompt.format(strategy=s), i) for i, s in enumerate(strategies)]
            
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=workers)

        detailed_results = []
        valid_answers = []
        futures = []
        
        try:
            for i, (system_prompt, attempt_index) in enumerate(tasks):
                future = executor.submit(
                    self._process_attempt, 
                    problem, 
                    system_prompt, 
                    temperatures[i],
                    attempt_index, 
                    stop_event, 
                    deadline
                )
                
                futures.append(future)
                
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)
                    if counts and counts[0][1] >= early_stop:
                        stop_event.set()
                        
                        for f in futures:
                            f.cancel()
                            
                        break

                except Exception as exc:
                    print(f'### Future failed: {exc}')
                    continue
        finally:
            if not stop_event.is_set():
                stop_event.set()
            try:
                executor.shutdown(wait=True, cancel_futures=True)
            except TypeError:
                executor.shutdown(wait=True)
    
            time.sleep(0.05)

        return detailed_results, valid_answers


    def _compute_times(self):
        start_time = time.time()
    
        elapsed_global = start_time - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
        
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
        
        deadline = time.time() + budget
        print(f'### Time left: {time_left:.1f} | Budget: {budget:.1f} | Deadline: {deadline:.2f}\n', flush=True)
        
        return start_time, deadline

        
    def solve_problem(self, problem: str) -> int:    
        start_time, deadline = self._compute_times()

        detailed_results, valid_answers = self.process_attempt_loop(
            attempts=self.attempts,
            problem=problem,
            system_prompt=self.cfg.system_prompt,
            temperatures=self.temperatures,
            workers=self.cfg.workers,
            early_stop=self.cfg.early_stop,
            deadline=deadline,
        )

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64', copy=False)
            display(results_dataframe)

        self.problems_remaining = max(0, self.problems_remaining - 1)

        elapsed = time.time() - start_time
        time_duration = f'{int(elapsed // 60):02d}:{int(elapsed % 60):02d}'
        print(f'\n### Time elapsed: {time_duration}\n')
            
        if not valid_answers:
            print(f'\nResult: 0\n')
            return 0

        final_answer = self._select_answer(detailed_results)
        
        return final_answer


    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
            
        if hasattr(self, 'log_file'):
            self.log_file.close()
            
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                    
                except Exception:
                    pass

In [9]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)
    
    print(f'\n=============================================================== ID: {id_value} ===============================================================', flush=True)
    
    try:
        if answer is not None:
            print(f"### Ground Truth: {answer.item(0)}\n", flush=True)
    except Exception as e:
        pass
    
    print(f'### Problem: {question_text}\n', flush=True)

    gc.disable()
    
    final_answer = solver.solve_problem(question_text)

    try:
        if answer is not None:
            print(f"\n### Final result | Ground Truth: {answer.item(0)} | Predicted: {final_answer}", flush=True)
    except Exception as e:
        pass
        
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [10]:
solver = AIMO3Solver(CFG)

### Loading model weights from /kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-35b-a3b/1 into OS Page Cache...
### Processed 74 files (143.85 GB) in 11.14 seconds.

### Waiting for vLLM server...
### Server is ready (took 0.00 seconds).



In [11]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        #('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
    )


=============================================================== ID: dd7f5e ===============================================================
### Ground Truth: 160

### Problem: Let $\mathcal{F}$ be the set of functions $\alpha \colon \mathbb{Z}\to \mathbb{Z}$ for which there are only finitely many $n \in \mathbb{Z}$ such that $\alpha(n) \neq 0$. 

For two functions $\alpha$ and $\beta$ in $\mathcal{F}$, define their product $\alpha\star\beta$ to be $\sum\limits_{n\in\mathbb{Z}} \alpha(n)\cdot \beta(n)$. Also, for $n\in\mathbb{Z}$, define a shift operator $S_n \colon \mathcal{F}\to \mathcal{F}$ by $S_n(\alpha)(t)=\alpha(t+n)$ for all $t \in \mathbb{Z}$.

A function $\alpha \in \mathcal{F}$ is called \emph{shifty} if 
\begin{itemize}
    \item $\alpha(m)=0$ for all integers $m<0$ and $m>8$ and
    \item There exists $\beta \in \mathcal{F}$ and integers $k \neq l$ such that for all $n \in \mathbb{Z}$
    \begin{equation*}
        S_n(\alpha)\star\beta =
        \begin{cases}
            1 

2026-03-24 15:23:03,871 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57627
2026-03-24 15:23:03,882 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57623
2026-03-24 15:23:03,893 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57614
2026-03-24 15:23:03,893 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57623
2026-03-24 15:23:03,893 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57610
2026-03-24 15:23:03,907 - base.py - 780 - INFO - ALL tokens: 363, Available tokens: 57577
2026-03-24 15:27:25,489 - local_code_interpreter.py - 234 - INFO - INFO: kernel process's PID = 4591
2026-03-24 15:27:27,610 - local_code_interpreter.py - 105 - INFO - stdout:

```
Exception reporting mode: Minimal

```
2026-03-24 15:27:28,031 - base.py - 780 - INFO - ALL tokens: 729, Available tokens: 57577
2026-03-24 15:28:54,751 - base.py - 780 - INFO - ALL tokens: 1135, Available tokens: 57577
2026-03-24 15:29:37,589 - local_code_interpreter.py - 234 -

- Attempt index 4 | Answer NOT available: Text 




2026-03-24 15:31:25,518 - base.py - 780 - INFO - ALL tokens: 2111, Available tokens: 57614


- Attempt index 5 | Answer NOT available: Text 


- Attempt index 1 | Answer NOT available: Text 




2026-03-24 15:32:20,446 - base.py - 780 - INFO - ALL tokens: 2669, Available tokens: 57623
2026-03-24 15:32:24,210 - base.py - 780 - INFO - ALL tokens: 1135, Available tokens: 57577
2026-03-24 15:32:27,660 - base.py - 780 - INFO - ALL tokens: 2638, Available tokens: 57614
2026-03-24 15:33:04,963 - base.py - 780 - INFO - ALL tokens: 3084, Available tokens: 57614
2026-03-24 15:33:09,594 - base.py - 780 - INFO - ALL tokens: 1715, Available tokens: 57577
2026-03-24 15:33:09,952 - base.py - 780 - INFO - ALL tokens: 3184, Available tokens: 57623
2026-03-24 15:33:19,382 - base.py - 780 - INFO - ALL tokens: 1715, Available tokens: 57577
2026-03-24 15:34:02,161 - base.py - 780 - INFO - ALL tokens: 3184, Available tokens: 57623
2026-03-24 15:34:05,707 - base.py - 780 - INFO - ALL tokens: 2026, Available tokens: 57577
2026-03-24 15:34:12,284 - base.py - 780 - INFO - ALL tokens: 3084, Available tokens: 57614
2026-03-24 15:34:13,518 - base.py - 780 - INFO - ALL tokens: 2245, Available tokens: 57577

KeyboardInterrupt: 